### **공통 준비**

In [1]:
!pip -q install -U ultralytics scikit-learn torch torchvision

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 64.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 115.7 MB/s eta 0:00:00


In [2]:
import os
import glob
import json
import random
import shutil
import hashlib
import kagglehub
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.image as mpimg

import torch
import ultralytics
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
print("Ultralytics:", ultralytics.__version__)
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(), "| Device count:", torch.cuda.device_count())

Ultralytics: 8.3.193
Torch: 2.8.0+cu126 | CUDA: True | Device count: 1


## **학습**

### **데이터셋 준비**

In [7]:
os.environ["KAGGLEHUB_CACHE"] = "/content"

# 다운로드
path = kagglehub.dataset_download("sujallimje/plant-pathogens")
print("Dataset downloaded to:", path)

Using Colab cache for faster access to the 'plant-pathogens' dataset.
Dataset downloaded to: /kaggle/input/plant-pathogens


In [5]:
RAW_ROOT   = Path("/content/datasets/sujallimje/plant-pathogens/versions/1/pathogen")
DATA_SPLIT = Path("/content/data_split")

os.makedirs(DATA_SPLIT, exist_ok=True)

In [6]:
VALID_NAMES = {"bacteria", "fungus", "healthy", "pests", "virus"}
cls_dirs = [p for p in RAW_ROOT.iterdir() if p.is_dir() and p.name.lower() in VALID_NAMES]
assert cls_dirs, f"클래스 폴더를 찾지 못했습니다: {RAW_ROOT}"

# 3) 분할 비율
train_ratio, val_ratio, test_ratio = 0.8, 0.1, 0.1
assert abs(train_ratio + val_ratio + test_ratio - 1.0) < 1e-6

def copy_list(files, out_dir: Path):
    out_dir.mkdir(parents=True, exist_ok=True)
    for f in files:
        dst = out_dir / Path(f).name
        if not dst.exists():  # 중복 방지
            shutil.copy2(f, dst)

# 4) 클래스별로 이미지 파일 모아 섞고 → train/val/test로 복사
for cls_dir in cls_dirs:
    cls = cls_dir.name.lower()

    files = []
    for ext in ("*.jpg","*.jpeg","*.png","*.bmp","*.webp"):
        files += glob.glob(str(cls_dir / ext))
    files = sorted(files)
    random.shuffle(files)

    n = len(files)
    if n == 0:
        print(f"⚠️  '{cls}' 폴더에 이미지가 없습니다. 건너뜁니다.")
        continue

    n_train = int(n * train_ratio)
    n_val   = int(n * val_ratio)
    train_files = files[:n_train]
    val_files   = files[n_train:n_train + n_val]
    test_files  = files[n_train + n_val:]

    copy_list(train_files, DATA_SPLIT / "train" / cls)
    copy_list(val_files,   DATA_SPLIT / "val"   / cls)
    copy_list(test_files,  DATA_SPLIT / "test"  / cls)

print("✅ 분할 완료:", DATA_SPLIT)
print("train 클래스 목록:", [p.name for p in (DATA_SPLIT / "train").iterdir()])

FileNotFoundError: [Errno 2] No such file or directory: '/content/datasets/sujallimje/plant-pathogens/versions/1/pathogen'

### **모델 학습**

In [ ]:
model = YOLO("yolov8n-cls.pt")  # ImageNet 사전학습 분류 가중치

results = model.train(
    data=str(DATA_SPLIT),   # data 루트(여기에 train/val/test가 포함돼야 함)
    epochs=10,
    imgsz=224,              # 분류 기본 224 추천(리소스 여유 있으면 320/384로)
    batch=32,               # Colab T4/V100 환경에 맞춰 조정
    optimizer="AdamW",
    lr0=1e-3,
    patience=10,
    verbose=False
)

### **검증**

In [ ]:
metrics = model.val(split="val")
print("📊 Validation metrics:", metrics)

### **테스트 평가**

In [ ]:
test_metrics = model.val(split="test")
print("📊 Test metrics:", test_metrics)

## **최종 검사**

In [ ]:
# 1) 학습된 best 가중치 불러오기
model = YOLO("/content/runs/classify/train/weights/best.pt")

# 2) 테스트할 임의의 이미지 경로
test_image = "/content/unnamed.jpg"

# 3) 예측
pred = model.predict(test_image, imgsz=224)  # imgsz는 학습 때와 동일하게

print("=== 예측 결과 ===")
print("파일:", test_image)

# top1 클래스와 확률
cls_id = pred[0].probs.top1
cls_name = pred[0].names[cls_id]
conf = float(pred[0].probs.top1conf)
print("예측 클래스:", cls_name)
print("확률:", conf)

# 4) 전체 클래스별 확률 보기
probs = {pred[0].names[i]: float(p) for i, p in enumerate(pred[0].probs)}
print("전체 확률:", probs)


image 1/1 /content/unnamed.jpg: 224x224 fungus 1.00, pests 0.00, bacteria 0.00, healthy 0.00, virus 0.00, 3.1ms
Speed: 3.3ms preprocess, 3.1ms inference, 0.2ms postprocess per image at shape (1, 3, 224, 224)
=== 예측 결과 ===
파일: /content/unnamed.jpg
예측 클래스: fungus


AttributeError: 'Probs' object has no attribute 'max'. See valid attributes below.

    A class for storing and manipulating classification probabilities.

    This class extends BaseTensor and provides methods for accessing and manipulating
    classification probabilities, including top-1 and top-5 predictions.

    Attributes:
        data (torch.Tensor | np.ndarray): The raw tensor or array containing classification probabilities.
        orig_shape (tuple | None): The original image shape as (height, width). Not used in this class.
        top1 (int): Index of the class with the highest probability.
        top5 (List[int]): Indices of the top 5 classes by probability.
        top1conf (torch.Tensor | np.ndarray): Confidence score of the top 1 class.
        top5conf (torch.Tensor | np.ndarray): Confidence scores of the top 5 classes.

    Methods:
        cpu: Return a copy of the probabilities tensor on CPU memory.
        numpy: Return a copy of the probabilities tensor as a nu